# IELTS Out-of-Domain Evaluation Prompt Matrix Construction

This notebook constructs the **Out-of-Domain Evaluation Prompt Matrix** used
to evaluate whether CEFR control generalizes beyond the EFCAMDAT prompt
distribution.

Fifty distinct IELTS Writing Task 2 prompts are sampled from the publicly
available `nlpatunt/D_Ielts_Writing_Task_2_Dataset` dataset on Hugging Face.
Each prompt is paired with all six CEFR target levels (A1–C2), producing a
balanced benchmark of **300 generation conditions (50 prompts × 6 CEFR levels)**.

The resulting dataset is used exclusively for generation-time evaluation and
is not included in the steering training data.

**Source dataset:**  
https://huggingface.co/datasets/nlpatunt/D_Ielts_Writing_Task_2_Dataset

In [ ]:
# =========================================================
# 0. SETUP DEPENDENCIES & AUTHENTICATION
# =========================================================
!pip install -q datasets pandas huggingface_hub

import random
import pandas as pd
from datasets import Dataset, load_dataset
from huggingface_hub import login, HfApi
from google.colab import userdata

# Define your target Hugging Face repository for the benchmark
HF_REPO_ID = "MohammadKhosravi/ielts-cefr-benchmark-prompts"

# Authenticate Hugging Face via Colab Secret HF_TOKEN
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
except Exception as e:
    print(f"⚠️ Secret HF_TOKEN not found in Colab secrets. Please make sure HF_TOKEN is set.")

# Set seed for reproducibility
random.seed(42)

# ==========================================
# 1. LOAD SOURCE DATASET & EXTRACT PROMPTS
# ==========================================
print("📥 Loading 'nlpatunt/D_Ielts_Writing_Task_2_Dataset' from Hugging Face...")
raw_ds = load_dataset("nlpatunt/D_Ielts_Writing_Task_2_Dataset", split="train")

# Extract unique, non-empty prompt strings
all_prompts = list(set([p.strip() for p in raw_ds['prompt'] if p and isinstance(p, str)]))
print(f"Found {len(all_prompts)} unique prompts in the source dataset.")

# Randomly select 50 distinct prompts
selected_prompts = random.sample(all_prompts, 50)
print(f"✅ Selected 50 unique random prompts.")

# ==========================================
# 2. GENERATE 300-ROW MULTI-LEVEL BENCHMARK
# ==========================================
cefr_levels = ["A1", "A2", "B1", "B2", "C1", "C2"]
benchmark_data = []

# Pair each of the 50 prompts with all 6 CEFR levels
for prompt_text in selected_prompts:
    for level in cefr_levels:
        benchmark_data.append({
            "prompt": prompt_text,
            "cefr": level
        })

df_benchmark = pd.DataFrame(benchmark_data)

print(f"\n📊 Generated Benchmark DataFrame Shape: {df_benchmark.shape}")
print("Sample entries:")
print(df_benchmark.head(12))

# Verify distribution
print("\nClass Counts Verification:")
print(df_benchmark['cefr'].value_counts())

# ==========================================
# 3. PUSH BENCHMARK DATASET TO HUGGING FACE
# ==========================================
print(f"\n🚀 Packaging and pushing dataset to HF Repo: {HF_REPO_ID}...")

# Convert Pandas DataFrame to Hugging Face Dataset format
hf_benchmark_ds = Dataset.from_pandas(df_benchmark)

try:
    hf_benchmark_ds.push_to_hub(
        repo_id=HF_REPO_ID,
        private=False,
        commit_message="Initial upload of 300-prompt out-of-domain IELTS Writing Task 2 CEFR benchmark dataset"
    )
    print(f"🎉 SUCCESS! Dataset successfully pushed to: https://huggingface.co/datasets/{HF_REPO_ID}")
except Exception as e:
    print(f"❌ Failed to push to Hugging Face Hub: {str(e)}")

📥 Loading 'nlpatunt/D_Ielts_Writing_Task_2_Dataset' from Hugging Face...


README.md:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.csv: reconstructing file:   0%|          |  0.00B / 40.0MB            

train.csv: downloading bytes:           |  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 6.38kB / 6.38kB            

🎉 SUCCESS! Dataset successfully pushed to: https://huggingface.co/datasets/MohammadKhosravi/ielts-cefr-benchmark-prompts
